# Strategy health check practice — 2026-10-02

This notebook reads the saved outputs from the independent frozen-parameter review. It presents reported metrics and diagnostics without rerunning strategy calculations. No investment conclusion is drawn here.

See the [review note](../../docs/notes/strategy-healthcheck-2026-10-02-review.md) and [output README](../../outputs/strategy-healthcheck-2026-10-02/README.md).


In [ ]:
import os
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "xquant").exists():
            return candidate
    raise RuntimeError("repository root not found")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / "outputs" / "strategy-healthcheck-2026-10-02"


In [ ]:
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display, Markdown
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        if isinstance(value, pd.DataFrame):
            print(f"DataFrame: {value.shape[0]} rows × {value.shape[1]} columns")
            print(value.head(5).to_string(index=False))
        else:
            print(value)

OUTPUT_DIR = Path("outputs/strategy-healthcheck-2026-10-02")

def show_csv(relative_path, title=None):
    path = OUTPUT_DIR / relative_path
    display(Markdown(f"### {title or relative_path}"))
    if not path.exists():
        display(Markdown(f"_Not available yet: `{path}`_"))
        return None
    frame = pd.read_csv(path)
    display(frame)
    return frame


## Frozen replay summaries

The summary files contain the strategy and period rows produced by the replay script. The table values below are loaded as saved.


In [ ]:
theme_summary = show_csv("replay/theme/summary.csv", "Theme replay summary")
btc_summary = show_csv("replay/btc/summary.csv", "BTC replay summary")
next_open_summary = show_csv("next_open/summary_core.csv", "Next-open core summary")

for filename in ("monthly_returns.csv", "monthly_stats.csv", "rolling_3y_5y.csv", "annual_returns.csv"):
    show_csv(f"next_open/{filename}")


## Return and cost tables

Monthly and annual series and cost sensitivity come from the frozen replay outputs. Next-open outputs currently provide their own core summary and execution audit; absent optional tables are reported as unavailable.


In [ ]:
for family in ("theme", "btc"):
    display(Markdown(f"## {family} replay"))
    for filename in ("monthly_returns.csv", "monthly_stats.csv", "rolling_3y_5y.csv", "annual_returns.csv", "cost_sensitivity.csv"):
        show_csv(f"replay/{family}/{filename}")


## Saved diagnostics

These diagnostics are loaded from the output directory. They are not recomputed in this notebook.


In [ ]:
for filename in (
    "data_coverage_and_revisions.csv",
    "C_prior_review_overlap.csv",
    "asset_observations.csv",
    "btc_correlations.csv",
):
    show_csv(filename)


## Optional results still in progress

If Bull10 or multi-asset full results have been written under this run directory, show their CSV tables here. Missing results remain clearly marked as unavailable.


In [ ]:
import gzip
import json
show_csv("next_open/summary_bull.csv")
show_csv("review_decisions.csv")
plain = OUTPUT_DIR / "multi_asset_full.json"
compressed = OUTPUT_DIR / "multi_asset_full.json.gz"
if plain.exists() or compressed.exists():
    if plain.exists():
        payload = json.loads(plain.read_text())
    else:
        with gzip.open(compressed, "rt") as stream:
            payload = json.load(stream)
    display(Markdown("### Multi-asset frozen-grid result"))
    display(payload["recommendation"])
    display(payload["coverage"])
    display(payload["stability"])
    display(pd.DataFrame([{ "strategy": row["id"], **row["summary"] } for row in payload["variants"]]))
else:
    display(Markdown("_Multi-asset result is not present._"))


In [ ]:
# Execution audits are presented verbatim from the generated JSON files.
import json

for relative_path in (
    "replay/theme/execution_audit.json",
    "replay/btc/execution_audit.json",
    "next_open/audit_core.json",
):
    path = OUTPUT_DIR / relative_path
    display(Markdown(f"### {relative_path}"))
    if path.exists():
        display(pd.DataFrame(json.loads(path.read_text())))
    else:
        display(Markdown(f"_Not available yet: `{path}`_"))
